# FIGS — Feature Dependence

Model-agnostic **permutation importance** (AUC drop when a feature column is
shuffled) and **partial-dependence** curves for the top features. Works with the
mlx-boosting wrapper without needing native importances.

In [1]:
# Run with the `met` conda env kernel.
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')   # so `import figs` works from notebooks/
from figs import config as C

In [ ]:
from figs.data.dataset import read_dataset, feature_columns
from figs.model.wrapper import GBDTModel
from figs.config import LEAD_BANDS
from sklearn.metrics import roc_auc_score
from pathlib import Path

import json
DATA='../Data/processed/figs_v2.parquet'; MODELS='../Data/models'  # DATA must match the models
HAZARD='tor'; BAND=LEAD_BANDS[0].name
# feature list/order from the models' feature_cols.json (matches what they expect)
feats = json.loads((Path(MODELS)/'feature_cols.json').read_text())
df = read_dataset(DATA)  # NOTE: loads the full matrix; heavy at ~5k features
missing = [c for c in feats if c not in df.columns]
if missing: raise ValueError(f'{len(missing)} model features absent from DATA; point DATA at the models\' training set')
va = df[df.split=='validation']
if 'fxx' in va:
    b=[x for x in LEAD_BANDS if x.name==BAND][0]; va=va[(va.fxx>=b.fmin)&(va.fxx<=b.fmax)]
va = va.sample(min(len(va), 40000), random_state=0)   # subsample for speed
m = GBDTModel.load(Path(MODELS)/f'hazard_{HAZARD}_{BAND}.pkl')
X = va[feats].to_numpy('float32'); y = va[HAZARD].to_numpy(int)
base = roc_auc_score(y, m.predict_pos(X)); print('baseline AUC', round(base,4))

## Permutation importance (top 25)

In [ ]:
rng = np.random.default_rng(0)
imp = {}
for j,f in enumerate(feats):
    Xp = X.copy(); Xp[:,j] = rng.permutation(Xp[:,j])
    imp[f] = base - roc_auc_score(y, m.predict_pos(Xp))
    if j % 100 == 0: print(f'{j}/{len(feats)}')
imp = pd.Series(imp).sort_values(ascending=False)
imp.head(25).iloc[::-1].plot.barh(figsize=(7,8)); plt.xlabel('AUC drop'); plt.title(f'{HAZARD} {BAND} permutation importance'); plt.show()
imp.head(25)

## Partial dependence for the top features

In [ ]:
top = imp.head(6).index.tolist()
fig, axes = plt.subplots(2,3, figsize=(15,8))
for ax,f in zip(axes.ravel(), top):
    j = feats.index(f)
    grid_vals = np.nanpercentile(X[:,j], np.linspace(2,98,20))
    pd_curve = []
    for g in grid_vals:
        Xt = X.copy(); Xt[:,j] = g; pd_curve.append(m.predict_pos(Xt).mean())
    ax.plot(grid_vals, pd_curve, 'o-'); ax.set_title(f); ax.set_ylabel('mean p')
plt.tight_layout(); plt.show()